<a href="https://colab.research.google.com/github/rlakshmi2k24/Learning_journey_1/blob/main/02_Claim_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xgboost
!pip install catboost
!pip install imbalanced-learn
!pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/sample_data/healthcare_analytical_dataset.csv")

print(df.shape)
df.head()

(25000, 26)


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approved_amount,claim_status,payment_days,billing_date,revenue_gap,senior_citizen_flag,long_stay_flag,payment_delay_flag,claim_rejected_flag,approval_ratio
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.00,Rejected,16.0,2025-06-18,23577.37,1,0,0,1,0.0
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,38178.81,Paid,18.0,2025-10-09,0.00,0,0,0,0,1.0
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,5038.97,Paid,NaN,2025-01-20,0.00,0,0,0,0,1.0
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,22813.34,Paid,16.0,2025-12-24,0.00,1,0,0,0,1.0
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,27106.95,Paid,14.0,2025-09-23,0.00,1,0,0,0,1.0


In [3]:
# ============================================================
# VISIT FREQUENCY
# ============================================================

visit_frequency = (
    df.groupby("patient_id")
      .size()
      .reset_index(name="visit_count")
)

# Drop existing 'visit_count' column if it exists to avoid MergeError on re-run
if 'visit_count' in df.columns:
    df = df.drop('visit_count', axis=1)

df = df.merge(
    visit_frequency,
    on="patient_id",
    how="left"
)

# ============================================================
# DEPARTMENT HIGH RISK RATE
# ============================================================

dept_risk = (
    df.groupby("department")["risk_score"]
      .apply(lambda x: (x=="High").mean())
      .reset_index(name="dept_high_risk_rate")
)

# Drop existing 'dept_high_risk_rate' column if it exists to avoid MergeError on re-run
if 'dept_high_risk_rate' in df.columns:
    df = df.drop('dept_high_risk_rate', axis=1)

df = df.merge(
    dept_risk,
    on="department",
    how="left"
)

# ============================================================
# INSURANCE REJECTION RATE
# ============================================================

provider_rejection = (
    df.groupby("insurance_provider")["claim_status"]
      .apply(lambda x: (x=="Rejected").mean())
      .reset_index(name="provider_rejection_rate")
)

# Drop existing 'provider_rejection_rate' column if it exists to avoid MergeError on re-run
if 'provider_rejection_rate' in df.columns:
    df = df.drop('provider_rejection_rate', axis=1)

df = df.merge(
    provider_rejection,
    on="insurance_provider",
    how="left"
)

# ============================================================
# REVENUE GAP %
# ============================================================

df["revenue_gap_pct"] = (
    (df["billed_amount"] - df["approved_amount"])
    /
    df["billed_amount"]
)

df["revenue_gap_pct"] = (
    df["revenue_gap_pct"]
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

# ============================================================
# LENGTH OF STAY BUCKET
# ============================================================

df["los_bucket"] = pd.cut(
    df["length_of_stay_hours"],
    bins=[0,12,24,48,1000],
    labels=[
        "Short",
        "Medium",
        "Long",
        "Critical"
    ]
)

# ============================================================
# SENIOR FLAG
# ============================================================

df["senior_flag"] = (
    df["age"] >= 60
).astype(int)

# ============================================================
# DOCTOR HIGH RISK RATE
# ============================================================

doctor_risk_rate = (
    df.groupby("doctor_id")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="doctor_high_risk_rate"
      )
)

# Drop existing 'doctor_high_risk_rate' column if it exists to avoid MergeError on re-run
if 'doctor_high_risk_rate' in df.columns:
    df = df.drop('doctor_high_risk_rate', axis=1)

df = df.merge(
    doctor_risk_rate,
    on="doctor_id",
    how="left"
)

# ============================================================
# AGE BUCKET
# ============================================================

df["age_bucket"] = pd.cut(
    df["age"],
    bins=[0,18,40,60,120],
    labels=[
        "Child",
        "Adult",
        "Middle",
        "Senior"
    ]
)

# ============================================================
# HIGH RISK CHRONIC
# ============================================================

df["high_risk_chronic"] = np.where(
    (df["chronic_flag"]==1)
    &
    (df["age"]>=60),
    1,
    0
)

# ============================================================
# PATIENT HIGH RISK RATE
# ============================================================

patient_high_risk_rate = (
    df.groupby("patient_id")
      ["risk_score"]
      .apply(
          lambda x:
          (x=="High").mean()
      )
      .reset_index(
          name="patient_high_risk_rate"
      )
)

# Drop existing 'patient_high_risk_rate' column if it exists to avoid MergeError on re-run
if 'patient_high_risk_rate' in df.columns:
    df = df.drop('patient_high_risk_rate', axis=1)

df = df.merge(
    patient_high_risk_rate,
    on="patient_id",
    how="left"
)

# ============================================================
# PATIENT AVERAGE LENGTH OF STAY
# ============================================================

patient_avg_los = (
    df.groupby("patient_id")
      ["length_of_stay_hours"]
      .mean()
      .reset_index(
          name="patient_avg_los"
      )
)

# Drop existing 'patient_avg_los' column if it exists to avoid MergeError on re-run
if 'patient_avg_los' in df.columns:
    df = df.drop('patient_avg_los', axis=1)

df = df.merge(
    patient_avg_los,
    on="patient_id",
    how="left"
)

In [4]:
# ============================================================
# VISIT FREQUENCY
# ============================================================

visit_frequency = (
    df.groupby("patient_id")
      .size()
      .reset_index(name="visit_count")
)

df = df.merge(
    visit_frequency,
    on="patient_id",
    how="left"
)

# ============================================================
# DEPARTMENT HIGH RISK RATE
# ============================================================

dept_risk = (
    df.groupby("department")["risk_score"]
      .apply(lambda x: (x=="High").mean())
      .reset_index(name="dept_high_risk_rate")
)

df = df.merge(
    dept_risk,
    on="department",
    how="left"
)

# ============================================================
# INSURANCE REJECTION RATE
# ============================================================

provider_rejection = (
    df.groupby("insurance_provider")["claim_status"]
      .apply(lambda x: (x=="Rejected").mean())
      .reset_index(name="provider_rejection_rate")
)

df = df.merge(
    provider_rejection,
    on="insurance_provider",
    how="left"
)

# ============================================================
# REVENUE GAP %
# ============================================================

df["revenue_gap_pct"] = (
    (df["billed_amount"] - df["approved_amount"])
    /
    df["billed_amount"]
)

df["revenue_gap_pct"] = (
    df["revenue_gap_pct"]
      .replace([np.inf, -np.inf], np.nan)
      .fillna(0)
)

# ============================================================
# LENGTH OF STAY BUCKET
# ============================================================

df["los_bucket"] = pd.cut(
    df["length_of_stay_hours"],
    bins=[0,12,24,48,1000],
    labels=[
        "Short",
        "Medium",
        "Long",
        "Critical"
    ]
)

# ============================================================
# SENIOR FLAG
# ============================================================

df["senior_flag"] = (
    df["age"] >= 60
).astype(int)

In [5]:
# ============================================================
# VISIT COUNT
# ============================================================

df["visit_count"] = (
    df.groupby("patient_id")["visit_id"]
      .transform("count")
)


# ============================================================
# DEPARTMENT HIGH RISK RATE
# ============================================================

df["dept_high_risk_rate"] = (
    df.groupby("department")["risk_score"]
      .transform(lambda x: (x == "High").mean())
)


# ============================================================
# PROVIDER REJECTION RATE
# ============================================================

df["provider_rejection_rate"] = (
    df.groupby("insurance_provider")["claim_status"]
      .transform(lambda x: (x == "Rejected").mean())
)

In [6]:
target = "claim_status"

features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "department",
    "visit_type",
    "risk_score",
    "length_of_stay_hours",
    "doctor_id",
    "billed_amount",
    "payment_days",
    "visit_count",
    "dept_high_risk_rate",
    "provider_rejection_rate",
    "revenue_gap_pct",
    "los_bucket",
    "senior_flag",
]

data = df[features + [target]].copy()

data = data.dropna()

X = data[features]
y = data[target]

In [7]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = X.select_dtypes(
    include=["object","category"]
).columns

X_encoded = X.copy()

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X_encoded[col] = le.fit_transform(
        X_encoded[col].astype(str)
    )

    encoders[col] = le

In [8]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X_encoded,
    y
)

mi_df = pd.DataFrame({
    "feature": X_encoded.columns,
    "score": mi_scores
})

mi_df = mi_df.sort_values(
    "score",
    ascending=False
)

print(mi_df)

                    feature     score
14          revenue_gap_pct  0.838946
10             payment_days  0.065674
9             billed_amount  0.038033
12      dept_high_risk_rate  0.006376
16              senior_flag  0.006107
2                      city  0.005536
8                 doctor_id  0.002824
5                visit_type  0.001605
6                risk_score  0.001560
0                       age  0.000267
1                    gender  0.000149
3        insurance_provider  0.000000
4                department  0.000000
7      length_of_stay_hours  0.000000
11              visit_count  0.000000
13  provider_rejection_rate  0.000000
15               los_bucket  0.000000


In [9]:
selected_features = (
    mi_df
    .head(15)
    ["feature"]
    .tolist()
)

X_selected = X_encoded[selected_features]

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [11]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(y_train.value_counts())

print(y_train_smote.value_counts())

claim_status
Paid        11585
Pending      4844
Rejected     2939
Name: count, dtype: int64
claim_status
Paid        11585
Rejected    11585
Pending     11585
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

rf_grid = {
    "n_estimators":[200,300,500],
    "max_depth":[5,10,15,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
    "max_features":["sqrt","log2"]
}

rf_search = RandomizedSearchCV(
    rf,
    rf_grid,
    n_iter=30,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    random_state=42
)

rf_search.fit(
    X_train_smote,
    y_train_smote
)

best_rf = rf_search.best_estimator_

In [14]:
print("Best Random Forest Parameters:")
print(rf_search.best_params_)

Best Random Forest Parameters:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}


In [15]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder # Added import

# Initialize LabelEncoder
label_encoder = LabelEncoder()
# Fit on the full target variable 'y' to ensure all possible classes are mapped
# 'y' is defined in cell 'psZr2OvFfyL3' as data[target] (claim_status string series)
label_encoder.fit(y)

# Encode y_train_smote to numerical labels
y_train_smote_encoded = label_encoder.transform(y_train_smote)

xgb = XGBClassifier(
    random_state=42,
    eval_metric="mlogloss",
    objective="multi:softprob",
    num_class=len(label_encoder.classes_) # Dynamically set num_class based on encoder
)

xgb_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_grid,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    error_score="raise"
)

xgb_search.fit(
    X_train_smote,
    y_train_smote_encoded # Corrected variable name
)

best_xgb = xgb_search.best_estimator_

print("Best XGBoost Parameters:")
print(xgb_search.best_params_)

Best XGBoost Parameters:
{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [16]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    verbose=0,
    random_state=42
)

cat_grid = {
    "depth":[4,6,8],
    "learning_rate":[0.01,0.05,0.1],
    "iterations":[200,500]
}

cat_search = RandomizedSearchCV(
    cat,
    cat_grid,
    n_iter=15,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    random_state=42
)

cat_search.fit(
    X_train_smote,
    y_train_smote
)

best_cat = cat_search.best_estimator_
print("Best CatBoost Parameters:")
print(cat_search.best_params_)


Best CatBoost Parameters:
{'learning_rate': 0.1, 'iterations': 500, 'depth': 8}


In [17]:
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

models = {
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "CatBoost": best_cat
}

results = []

# X_test is already processed (numerically encoded categorical features
# and filtered for selected features) from X_selected. Therefore,
# X_test_final_for_prediction can be directly assigned from X_test.
X_test_final_for_prediction = X_test.copy()

# Encode y_test to numerical labels using the same label_encoder
# that was fitted on 'y' (original string labels) in cell 'U-0i3cVK0XLk'.
y_test_encoded = label_encoder.transform(y_test)

for name, model in models.items():
    # Predict using the fully processed and feature-selected test set
    pred = model.predict(X_test_final_for_prediction)

    # Ensure pred is a 1D array for consistent processing
    if pred.ndim > 1 and pred.shape[1] == 1:
        pred = pred.flatten()

    # Ensure predictions are numerical for consistent evaluation with y_test
    # y_test is already numerically encoded (from y_encoded split).
    # Some models (like RandomForest, CatBoost) might predict string labels if trained on them,
    # while XGBoost predicts numerical labels if num_class is set.
    if isinstance(pred[0], str): # Check if the elements are string labels
        pred_encoded = label_encoder.transform(pred)
    else: # If predictions are already numerical (e.g., from XGBoost)
        pred_encoded = pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_encoded, pred_encoded),
        "F1 Macro": f1_score(
            y_test_encoded,
            pred_encoded,
            average="macro"
        )
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "F1 Macro",
    ascending=False
)

,Model,Accuracy,F1 Macro
0,Random Forest,0.978315,0.974283
1,XGBoost,0.978315,0.974203
2,CatBoost,0.977902,0.973462


In [18]:
from sklearn.metrics import classification_report

best_model = best_cat

# Use the pre-processed and feature-selected test set for prediction
pred_str = best_model.predict(X_test_final_for_prediction)

# Convert string predictions to numerical labels using the fitted label_encoder
# Flatten pred_str to a 1D array to avoid DataConversionWarning
# label_encoder was defined and fitted in cell 'U-0i3cVK0XLk'
pred = label_encoder.transform(pred_str.flatten())

print(
    classification_report(
        y_test_encoded, # Use the numerically encoded y_test
        pred
    )
)


              precision    recall  f1-score   support

           0       0.97      1.00      0.98      2896
           1       1.00      0.96      0.98      1211
           2       1.00      0.93      0.96       735

    accuracy                           0.98      4842
   macro avg       0.99      0.96      0.97      4842
weighted avg       0.98      0.98      0.98      4842



In [21]:
importance_df = pd.DataFrame({
    "Feature": X_selected.columns,
    "Importance": best_model.feature_importances_
})

importance_df = (
    importance_df
      .sort_values(
          "Importance",
          ascending=False
      )
)

print(
    importance_df.head(20)
)

                 Feature  Importance
0        revenue_gap_pct   49.355872
2          billed_amount    6.838620
1           payment_days    6.239694
6              doctor_id    5.235212
9                    age    4.871687
13  length_of_stay_hours    4.843366
14           visit_count    3.785654
5                   city    3.727031
3    dept_high_risk_rate    3.726239
11    insurance_provider    2.831896
12            department    2.817892
8             risk_score    2.485684
7             visit_type    2.028807
10                gender    0.981493
4            senior_flag    0.230854


In [23]:
import joblib

# Based on the F1 Macro score in results_df_2, best_cat has the highest score
best_model = best_cat
joblib.dump(
    best_model,
    "best_claim_prediction_model.pkl"
)

['best_claim_prediction_model.pkl']